***<h1>Transfermarket Dataset</h1>***

Guardamos en un dataset el nombre y valor de mercado unicamente de los jugadores en activo basandonos en last_season sea 2024 

In [21]:
import pandas as pd

# 1. CARGA DE DATOS
df_players = pd.read_csv('../data/raw/players.csv')
df_valuations = pd.read_csv('../data/raw/player_valuations.csv')

# 2. PREPARACIÓN DE VALORACIONES (Obtener la más reciente) ---

# Convertir la columna 'date' a datetime
df_valuations['date'] = pd.to_datetime(df_valuations['date'])

# Encontrar el índice de la valoración MÁS RECIENTE para CADA jugador
idx_max_date = df_valuations.groupby('player_id')['date'].idxmax()
df_latest_valuations = df_valuations.loc[idx_max_date].copy()

# Seleccionar solo ID y valor de mercado
df_values_only = df_latest_valuations[['player_id', 'market_value_in_eur']]


# 3. FILTRO DE JUGADORES ACTIVOS (last_season = 2024) ---

ACTIVE_SEASON = 2024

# Filtrar la tabla de jugadores original para obtener solo los IDs de los activos
df_active_players = df_players[df_players['last_season'] == ACTIVE_SEASON].copy()
active_player_ids = df_active_players['player_id']


# --- 4. FILTRO FINAL Y FUSIÓN ---

# Filtrar las valoraciones más recientes para mantener SOLO a los jugadores activos
df_final_filtered = df_values_only[
    df_values_only['player_id'].isin(active_player_ids)
].copy()

# Preparar la tabla de nombres
df_names = df_players[['player_id', 'name','date_of_birth']]

# Extraer solo el año de nacimiento (columna 'Born' en el dataset combinado)
df_names['Born'] = pd.to_datetime(df_names['date_of_birth']).dt.year 
# Convertir a entero y manejar nulos (por si acaso)
df_names['Born'] = df_names['Born'].fillna(-1).astype(int)

# Fusionar con la tabla de nombres
df_output = pd.merge(
    df_final_filtered,
    df_names[['player_id', 'name', 'Born']],
    on='player_id',
    how='left'
)

# 5. RESULTADO FINAL (Seleccionando solo las columnas requeridas)
df_final_dataset = df_output[['name', 'market_value_in_eur', 'Born']]

print(f"Número de jugadores activos con valoración más reciente: {len(df_final_dataset)}")
print("Primeras 5 filas del dataset final:")
print(df_final_dataset.head())

# 6. EXPORTACIÓN DEL PRODUCTO FINAL
df_final_dataset.to_csv('../data/raw/market_values.csv', index=False)

Número de jugadores activos con valoración más reciente: 6296
Primeras 5 filas del dataset final:
                  name  market_value_in_eur  Born
0         James Milner              1000000  1986
1  Anastasios Tsokanis               300000  1991
2        Jonas Hofmann              3000000  1992
3           Pepe Reina               600000  1982
4        Lionel Carole               400000  1991


C:\Users\yerayhurdra\AppData\Local\Temp\ipykernel_9288\1487882268.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_names['Born'] = pd.to_datetime(df_names['date_of_birth']).dt.year
C:\Users\yerayhurdra\AppData\Local\Temp\ipykernel_9288\1487882268.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_names['Born'] = df_names['Born'].fillna(-1).astype(int)


***<h1>Stats Dataset</h1>***

In [22]:
# 1. CARGA DE DATOS
df_historic = pd.read_csv('../data/raw/All_Players_1992-2025.csv')
df_actual = pd.read_csv('../data/raw/Season_2025-2026.csv')

# 2. CONCATENACIÓN DE LOS DATAFRAMES
df_combined = pd.concat([df_historic, df_actual], ignore_index=True)
print("\n--- Resultado de la Concatenación ---")
print(f"Total de filas combinado: {len(df_combined)}")

print("\nPrimeras 5 filas del DataFrame combinado:")
print(df_combined.head())
print("\nInformación de tipos de datos y nulos:")
df_combined.info()

# 3. EXPORTACIÓN DEL DATAFRAME COMBINADO
df_combined.to_csv('../data/raw/All_Players_1992-2026.csv', index=False)




--- Resultado de la Concatenación ---
Total de filas combinado: 94899

Primeras 5 filas del DataFrame combinado:
   PlayerID              Player          Squad      League Nation    Pos  \
0         4  Alexander Strehmel      Stuttgart  Bundesliga    GER  DF,MF   
1         4  Alexander Strehmel      Stuttgart  Bundesliga    GER  DF,MF   
2         4  Alexander Strehmel   Unterhaching  Bundesliga    GER  DF,MF   
3         4  Alexander Strehmel   Unterhaching  Bundesliga    GER  DF,MF   
4         6     Alois Reinhardt  Bayern Munich  Bundesliga    GER     DF   

    Age    Born     Season    MP  ...  The Best FIFA Mens Player  \
0  24.0  1968.0  1992-1993  20.0  ...                        0.0   
1  25.0  1968.0  1993-1994  13.0  ...                        0.0   
2  31.0  1968.0  1999-2000  25.0  ...                        0.0   
3  32.0  1968.0  2000-2001  32.0  ...                        0.0   
4  30.0  1961.0  1992-1993   5.0  ...                        0.0   

   UEFA Best Player 

***<h1>Final Merge</h1>***

In [23]:
# 1. CARGA DE DATOS
df_stats = pd.read_csv('../data/raw/All_Players_1992-2026.csv')
df_market_values = pd.read_csv('../data/raw/market_values.csv')

# 2. FUSIÓN DE LOS DATAFRAMES
df_merged = pd.merge(
    left=df_stats,
    right=df_market_values,
    left_on=['Player', 'Born'],
    right_on=['name', 'Born'],
    how='inner'
)

df_merged.drop(columns=['name'], inplace=True)
print("\n--- Resultado de la Fusión ---")
print(f"Total de filas con valor de mercado coincidente: {len(df_merged)}")

# 3. EXPORTACIÓN DEL DATAFRAME FINAL
df_merged.to_csv('../data/raw/final_dataset.csv', index=False)




--- Resultado de la Fusión ---
Total de filas con valor de mercado coincidente: 15903
